In [2]:
# =========================
# 1) INSTALL + DRIVE
# =========================
!pip -q install -U \
  "transformers>=4.41,<4.58" accelerate bitsandbytes \
  "huggingface_hub>=0.34,<1.0" \
  sentence-transformers faiss-cpu tqdm pandas

from google.colab import drive
drive.mount("/content/drive")

# ⚠️ Chemin de tes JSON nettoyés (tu m’as donné celui-ci)
DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"

# Dossier où on sauvegarde l'index FAISS + les chunks
OUT_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/_artifacts_rag_cnrs"
import os
os.makedirs(OUT_DIR, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)

Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full
OUT_DIR : /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/_artifacts_rag_cnrs


In [3]:
# =========================
# 2) CHECK DATA
# =========================
from pathlib import Path

json_files = sorted([str(p) for p in Path(DATA_DIR).glob("*.json")])
print("Nb JSON:", len(json_files))
print("Exemples:", json_files[:5])
assert len(json_files) > 0, "Aucun JSON trouvé : vérifie DATA_DIR."

Nb JSON: 127
Exemples: ['/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/guide_candidat_2025.json', '/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/instituts_cnrs.json', '/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/page_001.json', '/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/page_002.json', '/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/page_003.json']


In [4]:
# =========================
# 3) OPEN ONE JSON
# =========================
import json

sample = json_files[0]
with open(sample, "r", encoding="utf-8") as f:
    obj = json.load(f)

print("Fichier:", sample)
print("Type:", type(obj))
if isinstance(obj, dict):
    print("Keys:", list(obj.keys())[:30])
elif isinstance(obj, list):
    print("Len:", len(obj))
    print("First keys:", list(obj[0].keys()) if obj else None)

Fichier: /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/guide_candidat_2025.json
Type: <class 'dict'>
Keys: ['source', 'url', 'source_file', 'pages', 'sections']


In [5]:
# =========================
# 4) UTILS: CLEAN TEXT SAFELY
# =========================
import re

def to_text(x):
    """Convertit n'importe quel type (str/list/dict/None) en texte propre."""
    if x is None:
        return ""
    if isinstance(x, str):
        s = x
    elif isinstance(x, (list, dict)):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [6]:
# =========================
# 5) EXTRACT PASSAGES FROM JSON FILES
# =========================
from tqdm import tqdm

def infer_kind(filename: str):
    fn = filename.lower()
    # adapte si tu as d'autres conventions
    if "guide" in fn:
        return "process_general"
    if "avantages" in fn or "accompagner" in fn or "instituts" in fn:
        return "general_career"
    if fn.startswith("page_"):
        return "concours"
    return "general_cnrs"

def extract_passages(obj, file_path: str):
    """
    Convertit un JSON en passages.
    On essaye d'extraire:
    - concours: doc complet dans "pages" ou texte principal
    - documents généraux: pages ou sections
    """
    fn = Path(file_path).name
    kind = infer_kind(fn)
    out = []

    # Cas dict
    if isinstance(obj, dict):
        # cas avec pages
        if "pages" in obj and isinstance(obj["pages"], list):
            for p in obj["pages"]:
                txt = to_text(p.get("text") if isinstance(p, dict) else p)
                if txt:
                    out.append({
                        "text": txt,
                        "source": obj.get("source_file", fn),
                        "section": f"Page {p.get('page', '')}".strip() if isinstance(p, dict) else "Page",
                        "doc_id": fn.replace(".json", ""),
                        "kind": kind
                    })
        else:
            # fallback: tout le dict en texte
            txt = to_text(obj)
            if txt:
                out.append({
                    "text": txt,
                    "source": obj.get("source_file", fn),
                    "section": "Document (json)",
                    "doc_id": fn.replace(".json", ""),
                    "kind": kind
                })

    # Cas list
    elif isinstance(obj, list):
        for i, it in enumerate(obj):
            txt = to_text(it)
            if txt:
                out.append({
                    "text": txt,
                    "source": fn,
                    "section": f"Item {i}",
                    "doc_id": fn.replace(".json", ""),
                    "kind": kind
                })

    return out

passages = []
for fp in tqdm(json_files):
    with open(fp, "r", encoding="utf-8") as f:
        obj = json.load(f)
    passages.extend(extract_passages(obj, fp))

print("✅ Passages total:", len(passages))
print("Kinds:", {k: sum(1 for p in passages if p["kind"] == k) for k in sorted(set(p["kind"] for p in passages))})
print("\nExemple passage:\n", passages[0]["doc_id"], "|", passages[0]["section"], "\n", passages[0]["text"][:300], "...")

100%|██████████| 127/127 [00:01<00:00, 82.37it/s]

✅ Passages total: 156
Kinds: {'concours': 122, 'general_career': 12, 'general_cnrs': 1, 'process_general': 21}

Exemple passage:
 guide_candidat_2025 | Page 1 
 CONCOURS EXTERNES DES PERSONNELS INGÉNIEURS ET TECHNICIENS Le guide du candidat et de la candidate Edition 2025 ...


In [7]:
# =========================
# 6) CHUNKING
# =========================
def chunk_text(text, chunk_size=1200, overlap=200):
    text = text.strip()
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

chunked = []
for p in passages:
    for ch in chunk_text(p["text"], chunk_size=1200, overlap=200):
        chunked.append({
            "text": ch,
            "source": p["source"],
            "section": p["section"],
            "doc_id": p["doc_id"],
            "kind": p["kind"]
        })

print("✅ Chunks total:", len(chunked))
print("Exemple chunk:\n", chunked[0]["doc_id"], "|", chunked[0]["section"], "\n", chunked[0]["text"][:300], "...")

✅ Chunks total: 837
Exemple chunk:
 guide_candidat_2025 | Page 1 
 CONCOURS EXTERNES DES PERSONNELS INGÉNIEURS ET TECHNICIENS Le guide du candidat et de la candidate Edition 2025 ...


In [8]:
# =========================
# 7) EMBEDDINGS + FAISS (STREAMING, NO VSTACK)
# =========================
import torch, gc
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

EMBED_ID = "BAAI/bge-m3"

# ⚠️ si tu avais un LLM chargé (pas le cas dans ce notebook propre), on libère
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", device)

embedder = SentenceTransformer(EMBED_ID, device=device)
dim = embedder.get_sentence_embedding_dimension()

index = faiss.IndexFlatIP(dim)  # cosine si embeddings normalisés

texts = [c["text"] for c in chunked]
batch_size = 16 if device == "cuda" else 32  # réduit en GPU pour éviter OOM

for i in tqdm(range(0, len(texts), batch_size)):
    v = embedder.encode(
        texts[i:i+batch_size],
        normalize_embeddings=True,
        show_progress_bar=False
    )
    v = np.asarray(v, dtype="float32")
    index.add(v)

print("✅ FAISS size:", index.ntotal, "dim:", dim)

Embedding device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

100%|██████████| 53/53 [00:50<00:00,  1.04it/s]

✅ FAISS size: 837 dim: 1024


In [9]:
# =========================
# 8) SAVE ARTIFACTS (DRIVE)
# =========================
import json

INDEX_PATH = os.path.join(OUT_DIR, "cnrs_faiss.index")
CHUNKS_PATH = os.path.join(OUT_DIR, "cnrs_chunks.jsonl")

faiss.write_index(index, INDEX_PATH)

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    for c in chunked:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("✅ Saved:")
print(" -", INDEX_PATH)
print(" -", CHUNKS_PATH)

✅ Saved:
 - /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/_artifacts_rag_cnrs/cnrs_faiss.index
 - /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/_artifacts_rag_cnrs/cnrs_chunks.jsonl


In [10]:
# =========================
# 9) RELOAD (VERIFY)
# =========================
import json

index = faiss.read_index(INDEX_PATH)

chunked = []
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        chunked.append(json.loads(line))

print("✅ Reloaded index size:", index.ntotal)
print("✅ Reloaded chunks:", len(chunked))

✅ Reloaded index size: 837
✅ Reloaded chunks: 837


In [12]:
# =========================
# 9bis) OPTIONAL: RERANKER (recommandé)
# =========================
from sentence_transformers import CrossEncoder
RERANK_ID = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(RERANK_ID, device="cpu")  # ✅ CPU pour éviter OOM
print("✅ Reranker loaded on CPU")

✅ Reranker loaded on CPU


In [13]:
# =========================
# 10) RETRIEVE (FAISS + RERANK)
# =========================
def retrieve(query, embedder, index, chunked, k=8, pre_k=60):
    # 1) rappel large FAISS
    qv = embedder.encode([query], normalize_embeddings=True)
    qv = np.asarray(qv, dtype="float32")
    scores, ids = index.search(qv, pre_k)

    cands = [chunked[int(i)] for i in ids[0] if int(i) >= 0]
    if not cands:
        return []

    # 2) rerank
    pairs = [(query, c["text"]) for c in cands]
    rr = reranker.predict(pairs)

    ranked = sorted(zip(rr, cands), key=lambda x: x[0], reverse=True)[:k]
    return [{**c, "score": float(s)} for s, c in ranked]

# Test retrieval
test_q = "conditions pour candidater au concours"
res = retrieve(test_q, embedder, index, chunked, k=5)
for r in res:
    print(f"{r['score']:.3f} | {r['doc_id']} | {r['source']} | {r['section']}")

0.982 | guide_candidat_2025 | Guide candidat 2025.pdf | Page 12
0.957 | guide_candidat_2025 | Guide candidat 2025.pdf | Page 11
0.875 | guide_candidat_2025 | Guide candidat 2025.pdf | Page 9
0.833 | guide_candidat_2025 | Guide candidat 2025.pdf | Page 6
0.797 | guide_candidat_2025 | Guide candidat 2025.pdf | Page 12


In [15]:
from huggingface_hub import notebook_login
notebook_login()

In [16]:
# =========================
# 11) FREE EMBEDDER + LOAD LLM
# =========================
import gc, torch

if "embedder" in globals():
    del embedder

gc.collect()
torch.cuda.empty_cache()


from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ✅ Choisis ton LLM
# - Si Llama 3.1 8B te fait OOM, commence par Gemma / ou un modèle plus petit
LLM_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # ou Gemma si tu préfères

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(LLM_ID, use_fast=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print("✅ LLM loaded")

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✅ LLM loaded


In [17]:
# =========================
# 12) GENERATION HELPER
# =========================
import torch

@torch.inference_mode()
def llm_chat(system, user, max_new_tokens=400, do_sample=False, temperature=0.2, top_p=0.9):
    messages = [
        {"role":"system","content":system},
        {"role":"user","content":user}
    ]
    enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    input_ids = enc["input_ids"].to(llm.device)
    attention_mask = enc["attention_mask"].to(llm.device)

    gen_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        do_sample=do_sample,
    )
    if do_sample:
        gen_kwargs.update(dict(temperature=temperature, top_p=top_p))

    out = llm.generate(**gen_kwargs)
    gen = out[0][input_ids.shape[-1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

In [18]:
# =========================
# 12bis) SMALL TALK / ROUTER
# =========================
import re

SMALLTALK = [
    (r"^(bonjour|salut|hello|bonsoir)\b",
     "Bonjour 👋 Je peux t’aider à trouver un concours ingénieur CNRS adapté à ton profil, "
     "ou à expliquer un concours précis (missions, postes, conditions, phases du concours)."),
    (r"\b(merci|thanks|thx)\b",
     "Avec plaisir 🙂 Si tu veux, donne-moi ton profil (compétences, domaine, localisation) et je t’oriente vers les concours pertinents."),
    (r"\b(au revoir|à bientôt|bye)\b",
     "Au revoir 👋 N’hésite pas à revenir si tu as d’autres questions sur les concours ou les carrières au CNRS."),
    (r"^\b(qui es-tu|c'est quoi)\b",
     "Je suis un assistant RAG : je réponds à partir de tes documents CNRS et je cite les sources.")
]

def is_smalltalk(q: str):
    qn = q.strip().lower()
    for pat, resp in SMALLTALK:
        if re.search(pat, qn):
            return resp
    return None

In [19]:
# =========================
# 13) RAG ANSWER FUNCTION
# =========================
# On recharge embedder sur CPU pour retrieval (léger et stable)
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(EMBED_ID, device="cpu")

SYSTEM = """Tu es un assistant RAG CNRS.
Règles:
- Réponds en français, ton neutre et professionnel.
- N'écris pas "désolé" et ne mentionne pas "contexte fourni".
- Utilise UNIQUEMENT le CONTEXTE.
- Si l'info exacte manque, donne l'intervalle disponible (périodes) et indique où trouver le détail (convocation, site...).
- Ajoute des citations [n] pour chaque info importante.
"""

def format_context(passages, max_chars=450):
    parts = []
    for i,p in enumerate(passages,1):
        excerpt = p["text"][:max_chars].strip() + ("..." if len(p["text"]) > max_chars else "")
        parts.append(f"[{i}] ({p['source']} — {p['section']} — {p['doc_id']}) {excerpt}")
    return "\n\n".join(parts)

def is_orientation(q: str) -> bool:
    ql = q.lower()
    return any(x in ql for x in [
        "quels concours", "quel concours", "me correspondent", "profil", "je suis", "je travaille", "je cherche"
    ])

def answer_rag(question, k=8, min_score=0.25):
    # 0) small talk
    st_resp = is_smalltalk(question)
    if st_resp:
        return st_resp

    # 1) seuil adaptatif
    local_min = 0.18 if is_orientation(question) else min_score

    passages = retrieve(question, embedder, index, chunked, k=k, pre_k=80)

    if not passages or passages[0]["score"] < local_min:
        return "Cette information n’est pas précisée dans les documents disponibles, je ne peux donc pas donner une valeur exacte.\n\nSources:\n- (sources trop faibles)"



    ctx = format_context(passages)

    prompt = f"""QUESTION:
{question}

CONTEXTE:
{ctx}

Réponds en t'appuyant uniquement sur le contexte. Si c'est partiel, ajoute une section 'Limites'."""
    ans = llm_chat(SYSTEM, prompt, max_new_tokens=450, do_sample=False)

    # Sources (top 5)
    src_lines = []
    for i,p in enumerate(passages[:3],1):
        src_lines.append(f"- [{i}] {p['source']} | {p['section']} | {p['doc_id']}")
    return ans.strip() + "\n\nSources:\n" + "\n".join(src_lines)

# test
print(answer_rag("Combien de postes sont disponibles pour le concours ingénieur biologiste en analyse de données ?"))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Il existe plusieurs concours pour des postes d'ingénieur biologiste, mais nous nous concentrerons sur ceux qui correspondent à l'emploi type "Ingenieure ou ingenieur biologiste en analyse de données".

Selon les informations disponibles, il y a au moins 3 concours avec des postes disponibles pour ce type d'emploi :

- Concours N° 1 : 3 postes disponibles [1]
- Concours N° 3 : 2 postes disponibles [3]
- Concours N° 5 : 1 poste disponible [8]

Il est possible qu'il y ait d'autres concours ou postes disponibles, mais ces informations ne sont pas accessibles dans le contexte fourni.

Limites : 
- Les informations disponibles ne permettent pas de connaître le nombre total de postes disponibles pour ce type d'emploi.
- Il est possible qu'il y ait d'autres concours ou postes disponibles qui ne sont pas accessibles dans le contexte fourni.

Sources:
- [1] page_001.html | Document (json) | page_001
- [2] page_002.html | Document (json) | page_002
- [3] page_003.html | Document (json) | page_003

In [20]:
print(answer_rag("Bonjour"))
print(answer_rag("Merci beaucoup"))
print(answer_rag("Au revoir"))

Bonjour 👋 Je peux t’aider à trouver un concours ingénieur CNRS adapté à ton profil, ou à expliquer un concours précis (missions, postes, conditions, phases du concours).
Avec plaisir 🙂 Si tu veux, donne-moi ton profil (compétences, domaine, localisation) et je t’oriente vers les concours pertinents.
Au revoir 👋 N’hésite pas à revenir si tu as d’autres questions sur les concours ou les carrières au CNRS.


In [21]:
q = "Combien de postes pour le concours ingénieur biologiste en analyse de données ?"
res = retrieve(q, embedder, index, chunked, k=5, pre_k=60)
for r in res:
    print(f"{r['score']:.3f} | {r['doc_id']} | {r['source']} | {r['section']}")

0.991 | page_001 | page_001.html | Document (json)
0.982 | page_002 | page_002.html | Document (json)
0.963 | page_003 | page_003.html | Document (json)
0.890 | page_004 | page_004.html | Document (json)
0.871 | page_008 | page_008.html | Document (json)


In [22]:
# =========================
# 14) SIMPLE CHAT LOOP
# =========================
def chat_loop():
    print("=== Chatbot RAG CNRS (Notebook) ===")
    print("Commandes: /quit\n")
    while True:
        q = input("Vous: ").strip()
        if not q:
            continue
        if q.lower() in ["/quit", "quit", "exit"]:
            print("Fin du chat.")
            break
        print("\nAssistant:\n" + answer_rag(q) + "\n")

chat_loop()

=== Chatbot RAG CNRS (Notebook) ===
Commandes: /quit

Vous: Je suis data scientist, quels concours me correspondent ?

Assistant:
Cette information n’est pas précisée dans les documents disponibles, je ne peux donc pas donner une valeur exacte.

Sources:
- (sources trop faibles)

Vous: quit
Fin du chat.


In [23]:
# =========================
# 15) EVAL HELPER: ADD MODEL ANSWERS TO CSV
# =========================
import pandas as pd
from tqdm import tqdm

IN_CSV = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/jeu_test.csv"
OUT_CSV = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/questions_with_model_answers.csv"

df = pd.read_csv(IN_CSV)
answers = []
for q in tqdm(df["Question"].fillna("").astype(str).tolist()):
    q = q.strip()
    if not q:
        answers.append("")
        continue
    try:
        answers.append(answer_rag(q))
    except Exception as e:
        answers.append(f"[ERROR] {type(e).__name__}: {e}")

df["model_answer"] = answers
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("✅ Saved:", OUT_CSV)

  8%|▊         | 4/48 [28:06<5:09:06, 421.51s/it]


KeyboardInterrupt: 